# Minimum Stay Thresholds & Booking Potential

**Original Question:** At what minimum stay length do cancellations or low bookings spike across regions?

_Exported from PlotStudio_

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


# PlotStudio stub — no-op outside the app
def add_to_workspace(df):
    pass


# Load source data
df_airbnb = pd.read_csv("data/df_airbnb.csv")

## Task 1: Create analysis-ready features and a validated sample to study how minimum stay policies relate to booking potential across listings and regions while properly handling outliers and missingness.

**Acceptance Criteria:** We have a clean working DataFrame with engineered variables (occupancy_potential, amenity_count, host segment flags), clear documentation of outlier handling for minimum_nights, and sanity checks on distributions and group sizes that confirm the sample and units are correctly defined for downstream analyses.

### 1.1: Engineer booking-potential proxy and key segmentation features from the raw listing data.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Start from the source data without dropping rows
if "df_airbnb" not in globals():
    raise NameError("df_airbnb is not available in the current namespace.")

df_airbnb_features = df_airbnb.copy()

# Retain/parse geographic and host-start columns
if "host_since" in df_airbnb_features.columns:
    df_airbnb_features["host_since_parsed"] = pd.to_datetime(
        df_airbnb_features["host_since"], errors="coerce"
    )

# Ensure minimum_nights is numeric for feature engineering
df_airbnb_features["minimum_nights_num"] = pd.to_numeric(
    df_airbnb_features["minimum_nights"], errors="coerce"
)

# Practical cap and empirical 99th percentile cap
minimum_nights_99_cap = df_airbnb_features["minimum_nights_num"].quantile(0.99)
minimum_nights_99_cap_value = (
    int(np.ceil(minimum_nights_99_cap)) if pd.notna(minimum_nights_99_cap) else 60
)
minimum_nights_99_cap_value = max(minimum_nights_99_cap_value, 1)

# Booking-potential proxies: higher when minimum stay is shorter
# Raw proxy uses the observed minimum_nights as-is, with missing retained as NaN
# Capped proxies reduce the influence of extreme long minimum-stay policies

df_airbnb_features["booking_potential_proxy_raw"] = (
    365.0 / df_airbnb_features["minimum_nights_num"]
)
df_airbnb_features.loc[
    df_airbnb_features["minimum_nights_num"] <= 0, "booking_potential_proxy_raw"
] = np.nan

df_airbnb_features["minimum_nights_cap_60"] = df_airbnb_features[
    "minimum_nights_num"
].clip(lower=1, upper=60)
df_airbnb_features["booking_potential_proxy_cap_60"] = (
    365.0 / df_airbnb_features["minimum_nights_cap_60"]
)

df_airbnb_features["minimum_nights_cap_99p"] = df_airbnb_features[
    "minimum_nights_num"
].clip(lower=1, upper=minimum_nights_99_cap_value)
df_airbnb_features["booking_potential_proxy_cap_99p"] = (
    365.0 / df_airbnb_features["minimum_nights_cap_99p"]
)

# Host segment flags
if "host_is_superhost" in df_airbnb_features.columns:
    if df_airbnb_features["host_is_superhost"].dtype == bool:
        df_airbnb_features["is_superhost_flag"] = df_airbnb_features[
            "host_is_superhost"
        ].astype(int)
    else:
        df_airbnb_features["is_superhost_flag"] = (
            df_airbnb_features["host_is_superhost"]
            .astype(str)
            .str.lower()
            .isin(["t", "true", "1", "yes", "y"])
            .astype(int)
        )
else:
    df_airbnb_features["is_superhost_flag"] = np.nan

if "host_total_listings_count" in df_airbnb_features.columns:
    df_airbnb_features["host_total_listings_count_num"] = pd.to_numeric(
        df_airbnb_features["host_total_listings_count"], errors="coerce"
    )
    df_airbnb_features["is_professional_host_flag"] = (
        df_airbnb_features["host_total_listings_count_num"] >= 3
    ).astype("float")
else:
    df_airbnb_features["host_total_listings_count_num"] = np.nan
    df_airbnb_features["is_professional_host_flag"] = np.nan

# Amenity count from semi-structured text
if "amenities" in df_airbnb_features.columns:
    amenities_as_str = df_airbnb_features["amenities"].astype("string")
    df_airbnb_features["amenity_count"] = amenities_as_str.fillna("").str.count(
        ","
    ) + amenities_as_str.fillna("").ne("").astype(int)
    df_airbnb_features.loc[
        amenities_as_str.isna() | amenities_as_str.eq(""), "amenity_count"
    ] = np.nan
else:
    df_airbnb_features["amenity_count"] = np.nan

# Row-count validation and compact reporting
print("Row-count validation:")
print(f"Original rows: {len(df_airbnb):,}")
print(f"Feature rows:   {len(df_airbnb_features):,}")
print(f"Rows preserved:  {len(df_airbnb_features) == len(df_airbnb)}")
print("")
print("New engineered columns:")
new_columns = [
    "minimum_nights_num",
    "booking_potential_proxy_raw",
    "minimum_nights_cap_60",
    "booking_potential_proxy_cap_60",
    "minimum_nights_cap_99p",
    "booking_potential_proxy_cap_99p",
    "is_superhost_flag",
    "host_total_listings_count_num",
    "is_professional_host_flag",
    "amenity_count",
    "host_since_parsed",
]
existing_new_columns = [col for col in new_columns if col in df_airbnb_features.columns]
print(existing_new_columns)
print("")
print("Engineered field dtypes:")
print(df_airbnb_features[existing_new_columns].dtypes.astype(str))
print("")
print("Preview of engineered fields:")
preview_columns = [
    col
    for col in ["city", "neighbourhood", "district"] + existing_new_columns
    if col in df_airbnb_features.columns
]
print(df_airbnb_features[preview_columns].head(5))
print("")
print(
    f"Empirical 99th-percentile cap used for minimum_nights: {minimum_nights_99_cap_value}"
)

add_to_workspace(df_airbnb_features)


### 1.2: Understand the global distribution of minimum_nights and booking-potential proxies, and identify the main ranges and outliers.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use the existing engineered dataframe; do not recreate or overwrite it.
df_profile_source = df_airbnb_features

summary_cols = [
    "minimum_nights_num",
    "minimum_nights_cap_60",
    "minimum_nights_cap_99p",
    "booking_potential_proxy_raw",
    "booking_potential_proxy_cap_60",
    "booking_potential_proxy_cap_99p",
]

# Combined descriptive table for minimum-stay and booking-potential proxy columns.
# Transpose after aggregating so each row is a metric/column combination.
df_profile_summary = (
    df_profile_source[summary_cols]
    .agg(
        [
            "count",
            "mean",
            "std",
            "min",
            lambda s: s.quantile(0.05),
            lambda s: s.quantile(0.25),
            "median",
            lambda s: s.quantile(0.75),
            lambda s: s.quantile(0.90),
            lambda s: s.quantile(0.95),
            lambda s: s.quantile(0.99),
            "max",
        ]
    )
    .T
)

df_profile_summary.columns = [
    "count",
    "mean",
    "std",
    "min",
    "5%",
    "25%",
    "50%",
    "75%",
    "90%",
    "95%",
    "99%",
    "max",
]

print(
    "Combined descriptive summary for minimum-stay and booking-potential proxy columns:"
)
print(df_profile_summary.round(3))
print("")

# Vectorized bracket profiling using cut categories and a separate 90+ flag.
mn_series = pd.to_numeric(df_profile_source["minimum_nights_num"], errors="coerce")

df_bracket_profile = pd.DataFrame(
    {
        "minimum_nights_num": mn_series,
        "bracket": pd.cut(
            mn_series,
            bins=[0, 1, 2, 3, 6, 7, 13, 14, 29, 30, 59, np.inf],
            labels=[
                "1 night",
                "2 nights",
                "3 nights",
                "4-6 nights",
                "7 nights",
                "8-13 nights",
                "14 nights",
                "15-29 nights",
                "30 nights",
                "31-59 nights",
                "60+ nights",
            ],
            right=True,
            include_lowest=False,
        ),
    }
)

df_bracket_counts = (
    df_bracket_profile["bracket"]
    .value_counts(dropna=False)
    .reindex(
        [
            "1 night",
            "2 nights",
            "3 nights",
            "4-6 nights",
            "7 nights",
            "8-13 nights",
            "14 nights",
            "15-29 nights",
            "30 nights",
            "31-59 nights",
            "60+ nights",
        ]
    )
)

total_nonmissing = int(mn_series.notna().sum())

df_bracket_summary = pd.DataFrame(
    {
        "bracket": df_bracket_counts.index,
        "count": df_bracket_counts.fillna(0).astype(int).values,
    }
)
df_bracket_summary["share_pct"] = np.where(
    total_nonmissing > 0, df_bracket_summary["count"] / total_nonmissing * 100.0, np.nan
)

# 90+ nights share, computed separately and vectorized.
count_90_plus = int((mn_series >= 90).sum())
share_90_plus = (
    (count_90_plus / total_nonmissing * 100.0) if total_nonmissing > 0 else np.nan
)

print("Key minimum-stay brackets:")
print(
    df_bracket_summary.to_string(index=False, formatters={"share_pct": "{:.2f}".format})
)
print("")
print(
    f"90+ nights: {count_90_plus:,} listings ({share_90_plus:.2f}% of non-missing minimum-stay values)"
)
print("")

cap_60_share = df_profile_source["minimum_nights_num"].gt(60).mean() * 100.0
cap_99_value = int(
    pd.to_numeric(df_profile_source["minimum_nights_cap_99p"], errors="coerce").max()
)
cap_99_share = df_profile_source["minimum_nights_num"].gt(cap_99_value).mean() * 100.0

print(
    f"Practical caps: 60 nights is a stable cutoff because {cap_60_share:.2f}% of listings exceed it; "
    f"the empirical 99th-percentile cap is {cap_99_value} nights, with {cap_99_share:.2f}% of listings above that threshold."
)


### 1.3: Visually inspect how minimum_nights relates to the booking-potential proxy and where most listings cluster.
_Output: figure_

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Use the existing engineered dataframe; do not recreate or overwrite it.
df_plot_source = df_airbnb_features.loc[
    df_airbnb_features["minimum_nights_num"].between(1, 60)
].copy()

# Keep integer minimum-stay values only for the readable 1-60 window.
df_plot_source["minimum_nights_int"] = (
    pd.to_numeric(df_plot_source["minimum_nights_num"], errors="coerce")
    .round()
    .astype("Int64")
)

df_plot_source = df_plot_source.loc[
    df_plot_source["minimum_nights_int"].between(1, 60)
].copy()

# Median booking potential and dominant city for each minimum-stay value.
df_median_by_mn = (
    df_plot_source.groupby("minimum_nights_int", as_index=False)
    .agg(
        median_booking_potential=("booking_potential_proxy_cap_60", "median"),
        dominant_city=(
            "city",
            lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0],
        ),
        city_count=("city", "count"),
    )
    .sort_values("minimum_nights_int")
)

df_median_by_mn["minimum_nights_int"] = df_median_by_mn["minimum_nights_int"].astype(
    int
)

fig = px.line(
    df_median_by_mn,
    x="minimum_nights_int",
    y="median_booking_potential",
    color="dominant_city",
    markers=True,
    title="Booking potential drops rapidly as minimum stay increases (global view)",
    labels={
        "minimum_nights_int": "Minimum stay (nights)",
        "median_booking_potential": "Median booking potential proxy (cap at 60 nights)",
        "dominant_city": "Dominant city",
    },
)
fig.update_traces(mode="lines+markers", marker_line_color="black", marker_line_width=1)
fig.update_yaxes(tickformat=",")
fig.show()


### 1.4: Validate that the analysis uses the correct unit (listing) and that regional and segment groups have sufficient sample sizes for comparisons.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use the existing engineered dataframe; do not recreate or overwrite it.
df_validation_source = df_airbnb_features

print("Listing identifier validation:")
print(f"Unique listing_id values: {df_validation_source['listing_id'].nunique():,}")
print(f"Total rows:              {len(df_validation_source):,}")
print(
    f"listing_id is unique:     {df_validation_source['listing_id'].nunique() == len(df_validation_source)}"
)
print("")

# Listings per city, sorted descending.
df_listings_by_city = (
    df_validation_source.groupby("city", as_index=False)
    .agg(listing_count=("listing_id", "count"))
    .sort_values("listing_count", ascending=False)
)
print("Listings per city:")
print(df_listings_by_city.to_string(index=False))
print("")

# City-by-room-type count table.
df_city_room_counts = (
    df_validation_source.groupby(["city", "room_type"])
    .size()
    .reset_index(name="listing_count")
    .pivot(index="city", columns="room_type", values="listing_count")
    .fillna(0)
    .astype(int)
)
df_city_room_counts = df_city_room_counts.loc[
    df_city_room_counts.sum(axis=1).sort_values(ascending=False).index
]
df_city_room_counts = df_city_room_counts[
    df_city_room_counts.sum(axis=0).sort_values(ascending=False).index
]
print("City by room type count table:")
print(df_city_room_counts.to_string())
print("")

# Overall counts for host flags.
df_host_flag_overall = pd.DataFrame(
    {
        "flag": ["is_superhost_flag", "is_professional_host_flag"],
        "count_1": [
            int((df_validation_source["is_superhost_flag"] == 1).sum()),
            int((df_validation_source["is_professional_host_flag"] == 1).sum()),
        ],
        "count_0": [
            int((df_validation_source["is_superhost_flag"] == 0).sum()),
            int((df_validation_source["is_professional_host_flag"] == 0).sum()),
        ],
        "missing_or_other": [
            int(df_validation_source["is_superhost_flag"].isna().sum()),
            int(df_validation_source["is_professional_host_flag"].isna().sum()),
        ],
    }
)
print("Overall host flag counts:")
print(df_host_flag_overall.to_string(index=False))
print("")

# City-level counts for host flags.
df_city_host_flags = (
    df_validation_source.assign(
        superhost_present=(df_validation_source["is_superhost_flag"] == 1).astype(int),
        professional_host_present=(
            df_validation_source["is_professional_host_flag"] == 1
        ).astype(int),
    )
    .groupby("city", as_index=False)
    .agg(
        total_listings=("listing_id", "count"),
        superhost_count=("superhost_present", "sum"),
        professional_host_count=("professional_host_present", "sum"),
    )
    .sort_values("total_listings", ascending=False)
)
print("City-level host flag counts:")
print(df_city_host_flags.to_string(index=False))
print("")

# Main practical analysis window: minimum stay at or below 60 nights.
df_city_within_60 = (
    df_validation_source.loc[df_validation_source["minimum_nights_num"] <= 60]
    .groupby("city", as_index=False)
    .agg(listings_within_60=("listing_id", "count"))
    .sort_values("listings_within_60", ascending=False)
)
print(
    "Listings per city within the main practical analysis window (minimum stay <= 60 nights):"
)
print(df_city_within_60.to_string(index=False))
print("")

# Flag thin city-room-type combinations for caution.
df_city_room_thin = (
    df_validation_source.groupby(["city", "room_type"])
    .size()
    .reset_index(name="listing_count")
    .sort_values("listing_count", ascending=True)
)
df_thin_city_room = df_city_room_thin.loc[df_city_room_thin["listing_count"] <= 100]

if len(df_thin_city_room) > 0:
    print(
        "Caution: thin city-by-room-type combinations with very low counts (<= 100 listings):"
    )
    print(df_thin_city_room.to_string(index=False))
else:
    print(
        "No city-by-room-type combinations fall below the low-count caution threshold of 100 listings."
    )


## Task 2: Identify global tipping points in minimum stay length where booking potential sharply declines, using the engineered proxies and focusing on interpretable breakpoints along the minimum_nights scale.

**Acceptance Criteria:** We have quantitative and visual evidence of specific minimum_nights thresholds (e.g., 2 vs 3, 3 vs 4–6, 7, 14, 30) where occupancy_potential distributions change markedly, including effect sizes and sample sizes, and at least one exploratory breakpoint/segmented analysis confirming or challenging these thresholds at the global level.

_Mode: exploratory_

### 2.1: Summarize booking potential statistics by each distinct minimum_nights value within a practical range to spot candidate thresholds.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use the existing engineered dataframe; do not recreate or overwrite it.
df_min_nights_source = df_airbnb_features.loc[
    df_airbnb_features["minimum_nights_num"].le(60)
].copy()

# Keep only integer minimum-stay values in the practical analysis window.
df_min_nights_source["minimum_nights_int"] = (
    pd.to_numeric(df_min_nights_source["minimum_nights_num"], errors="coerce")
    .round()
    .astype("Int64")
)

df_min_nights_source = df_min_nights_source.loc[
    df_min_nights_source["minimum_nights_int"].between(1, 60)
].copy()

# Aggregate booking-potential proxy statistics by each integer minimum stay.
df_min_nights_summary = (
    df_min_nights_source.groupby("minimum_nights_int", as_index=False)
    .agg(
        listing_count=("listing_id", "count"),
        mean_booking_potential=("booking_potential_proxy_cap_60", "mean"),
        median_booking_potential=("booking_potential_proxy_cap_60", "median"),
        p10_booking_potential=(
            "booking_potential_proxy_cap_60",
            lambda s: s.quantile(0.10),
        ),
        p25_booking_potential=(
            "booking_potential_proxy_cap_60",
            lambda s: s.quantile(0.25),
        ),
        p75_booking_potential=(
            "booking_potential_proxy_cap_60",
            lambda s: s.quantile(0.75),
        ),
        p90_booking_potential=(
            "booking_potential_proxy_cap_60",
            lambda s: s.quantile(0.90),
        ),
    )
    .sort_values("minimum_nights_int")
    .reset_index(drop=True)
)

# Previous-value comparisons for median booking potential.
df_min_nights_summary["prev_median_booking_potential"] = df_min_nights_summary[
    "median_booking_potential"
].shift(1)
df_min_nights_summary["median_drop_abs"] = (
    df_min_nights_summary["prev_median_booking_potential"]
    - df_min_nights_summary["median_booking_potential"]
)
df_min_nights_summary["median_drop_pct"] = (
    df_min_nights_summary["median_drop_abs"]
    / df_min_nights_summary["prev_median_booking_potential"]
    * 100.0
)

print("Minimum stay summary in the practical analysis window (first part):")
print(
    df_min_nights_summary[
        [
            "minimum_nights_int",
            "listing_count",
            "mean_booking_potential",
            "median_booking_potential",
            "p10_booking_potential",
            "p25_booking_potential",
            "p75_booking_potential",
            "p90_booking_potential",
        ]
    ]
    .head(15)
    .round(2)
    .to_string(index=False)
)
print("")

selected_policy_values = [7, 14, 30]
df_selected_policy_values = df_min_nights_summary.loc[
    df_min_nights_summary["minimum_nights_int"].isin(selected_policy_values),
    [
        "minimum_nights_int",
        "listing_count",
        "mean_booking_potential",
        "median_booking_potential",
        "p10_booking_potential",
        "p25_booking_potential",
        "p75_booking_potential",
        "p90_booking_potential",
    ],
].copy()

print("Selected common policy values:")
print(df_selected_policy_values.round(2).to_string(index=False))
print("")

# Largest relative drops in median booking potential versus the previous minimum-stay value.
df_drop_ranking = df_min_nights_summary.loc[
    df_min_nights_summary["median_drop_pct"].notna(),
    ["minimum_nights_int", "median_drop_pct", "median_drop_abs", "listing_count"],
].sort_values(["median_drop_pct", "listing_count"], ascending=[False, False])

print(
    "Minimum-stay values with the largest relative drop in median booking potential versus the previous value:"
)
print(
    df_drop_ranking.head(10)
    .round({"median_drop_pct": 2, "median_drop_abs": 2})
    .to_string(index=False)
)
print("")

# Values with high listing counts and relatively low booking potential.
# Define "high listing counts" as the top quartile within the 1-60 window.
listing_count_threshold = df_min_nights_summary["listing_count"].quantile(0.75)
df_high_count_low_potential = df_min_nights_summary.loc[
    (df_min_nights_summary["listing_count"] >= listing_count_threshold)
    & (df_min_nights_summary["minimum_nights_int"].notna())
].copy()

# Rank by low median booking potential, then by higher listing counts.
df_high_count_low_potential = df_high_count_low_potential.sort_values(
    ["median_booking_potential", "listing_count"], ascending=[True, False]
)

print(
    "Minimum-stay values combining high listing counts with relatively low booking potential:"
)
print(
    df_high_count_low_potential[
        [
            "minimum_nights_int",
            "listing_count",
            "median_booking_potential",
            "mean_booking_potential",
            "p25_booking_potential",
            "p75_booking_potential",
        ]
    ]
    .head(10)
    .round(2)
    .to_string(index=False)
)
print("")

add_to_workspace(df_min_nights_summary)


### 2.2: Create an intuitive visual of how aggregated booking potential changes by minimum_nights, revealing clear threshold candidates.
_Output: figure_

In [ ]:
import plotly.express as px
import pandas as pd

# Use the existing aggregated summary; do not recreate it.
df_plot_min_stay = df_min_nights_summary.loc[
    df_min_nights_summary["minimum_nights_int"].between(1, 30)
].copy()

df_plot_min_stay["minimum_nights_int"] = pd.to_numeric(
    df_plot_min_stay["minimum_nights_int"], errors="coerce"
)
df_plot_min_stay["listing_count"] = pd.to_numeric(
    df_plot_min_stay["listing_count"], errors="coerce"
)
df_plot_min_stay["median_booking_potential"] = pd.to_numeric(
    df_plot_min_stay["median_booking_potential"], errors="coerce"
)

# Plotly line() does not support size for marker scaling, so use marker size on a scatter trace.
fig = px.scatter(
    df_plot_min_stay.sort_values("minimum_nights_int"),
    x="minimum_nights_int",
    y="median_booking_potential",
    size="listing_count",
    size_max=22,
    title="Global booking-potential proxy by minimum stay length",
    labels={
        "minimum_nights_int": "Minimum stay (nights)",
        "median_booking_potential": "Median booking potential proxy",
        "listing_count": "Listing count",
    },
)
fig.update_traces(mode="lines+markers", marker_line_color="black", marker_line_width=1)
fig.update_yaxes(tickformat=",")
fig.show()


### 2.3: Quantify whether there are one or more breakpoints in the relationship between minimum_nights and booking potential at the global level.
_Output: exploration_

In [ ]:
import numpy as np
import pandas as pd

# Use the existing aggregated summary; do not recreate or overwrite it.
df_breakpoint_source = df_min_nights_summary.loc[
    df_min_nights_summary["minimum_nights_int"].between(1, 60)
].copy()

df_breakpoint_source["minimum_nights_int"] = pd.to_numeric(
    df_breakpoint_source["minimum_nights_int"], errors="coerce"
)
df_breakpoint_source["listing_count"] = pd.to_numeric(
    df_breakpoint_source["listing_count"], errors="coerce"
)
df_breakpoint_source["median_booking_potential"] = pd.to_numeric(
    df_breakpoint_source["median_booking_potential"], errors="coerce"
)
df_breakpoint_source["mean_booking_potential"] = pd.to_numeric(
    df_breakpoint_source["mean_booking_potential"], errors="coerce"
)
df_breakpoint_source["median_drop_abs"] = pd.to_numeric(
    df_breakpoint_source["median_drop_abs"], errors="coerce"
)
df_breakpoint_source["median_drop_pct"] = pd.to_numeric(
    df_breakpoint_source["median_drop_pct"], errors="coerce"
)

# Consecutive-drop ranking: identifies sharp local changes in the monotonic decline.
df_drop_ranking = (
    df_breakpoint_source.loc[
        df_breakpoint_source["median_drop_pct"].notna(),
        [
            "minimum_nights_int",
            "listing_count",
            "median_booking_potential",
            "median_drop_abs",
            "median_drop_pct",
        ],
    ]
    .sort_values(["median_drop_pct", "listing_count"], ascending=[False, False])
    .reset_index(drop=True)
)

# Grouped-band comparisons: compare common policy bands to show whether the decline is smooth or threshold-like.
df_band_stats = pd.DataFrame(
    [
        {
            "band": "1-3 nights",
            "n_values": 3,
            "listing_count": int(
                df_breakpoint_source.loc[
                    df_breakpoint_source["minimum_nights_int"].between(1, 3),
                    "listing_count",
                ].sum()
            ),
            "mean_median_booking_potential": df_breakpoint_source.loc[
                df_breakpoint_source["minimum_nights_int"].between(1, 3),
                "median_booking_potential",
            ].mean(),
            "median_of_medians": df_breakpoint_source.loc[
                df_breakpoint_source["minimum_nights_int"].between(1, 3),
                "median_booking_potential",
            ].median(),
        },
        {
            "band": "4-6 nights",
            "n_values": 3,
            "listing_count": int(
                df_breakpoint_source.loc[
                    df_breakpoint_source["minimum_nights_int"].between(4, 6),
                    "listing_count",
                ].sum()
            ),
            "mean_median_booking_potential": df_breakpoint_source.loc[
                df_breakpoint_source["minimum_nights_int"].between(4, 6),
                "median_booking_potential",
            ].mean(),
            "median_of_medians": df_breakpoint_source.loc[
                df_breakpoint_source["minimum_nights_int"].between(4, 6),
                "median_booking_potential",
            ].median(),
        },
        {
            "band": "7-14 nights",
            "n_values": 8,
            "listing_count": int(
                df_breakpoint_source.loc[
                    df_breakpoint_source["minimum_nights_int"].between(7, 14),
                    "listing_count",
                ].sum()
            ),
            "mean_median_booking_potential": df_breakpoint_source.loc[
                df_breakpoint_source["minimum_nights_int"].between(7, 14),
                "median_booking_potential",
            ].mean(),
            "median_of_medians": df_breakpoint_source.loc[
                df_breakpoint_source["minimum_nights_int"].between(7, 14),
                "median_booking_potential",
            ].median(),
        },
        {
            "band": "15-29 nights",
            "n_values": 15,
            "listing_count": int(
                df_breakpoint_source.loc[
                    df_breakpoint_source["minimum_nights_int"].between(15, 29),
                    "listing_count",
                ].sum()
            ),
            "mean_median_booking_potential": df_breakpoint_source.loc[
                df_breakpoint_source["minimum_nights_int"].between(15, 29),
                "median_booking_potential",
            ].mean(),
            "median_of_medians": df_breakpoint_source.loc[
                df_breakpoint_source["minimum_nights_int"].between(15, 29),
                "median_booking_potential",
            ].median(),
        },
        {
            "band": "30-60 nights",
            "n_values": 31,
            "listing_count": int(
                df_breakpoint_source.loc[
                    df_breakpoint_source["minimum_nights_int"].between(30, 60),
                    "listing_count",
                ].sum()
            ),
            "mean_median_booking_potential": df_breakpoint_source.loc[
                df_breakpoint_source["minimum_nights_int"].between(30, 60),
                "median_booking_potential",
            ].mean(),
            "median_of_medians": df_breakpoint_source.loc[
                df_breakpoint_source["minimum_nights_int"].between(30, 60),
                "median_booking_potential",
            ].median(),
        },
    ]
)

df_band_stats["share_of_listings_pct"] = (
    df_band_stats["listing_count"] / df_band_stats["listing_count"].sum() * 100.0
)

df_band_comparison = pd.DataFrame(
    {
        "comparison": ["1-3 vs 4-6", "4-6 vs 7-14", "7-14 vs 15-29", "15-29 vs 30-60"],
        "median_drop_abs": [
            df_band_stats.loc[
                df_band_stats["band"] == "1-3 nights", "median_of_medians"
            ].iloc[0]
            - df_band_stats.loc[
                df_band_stats["band"] == "4-6 nights", "median_of_medians"
            ].iloc[0],
            df_band_stats.loc[
                df_band_stats["band"] == "4-6 nights", "median_of_medians"
            ].iloc[0]
            - df_band_stats.loc[
                df_band_stats["band"] == "7-14 nights", "median_of_medians"
            ].iloc[0],
            df_band_stats.loc[
                df_band_stats["band"] == "7-14 nights", "median_of_medians"
            ].iloc[0]
            - df_band_stats.loc[
                df_band_stats["band"] == "15-29 nights", "median_of_medians"
            ].iloc[0],
            df_band_stats.loc[
                df_band_stats["band"] == "15-29 nights", "median_of_medians"
            ].iloc[0]
            - df_band_stats.loc[
                df_band_stats["band"] == "30-60 nights", "median_of_medians"
            ].iloc[0],
        ],
    }
)
df_band_comparison["median_drop_pct"] = (
    df_band_comparison["median_drop_abs"]
    / [
        df_band_stats.loc[
            df_band_stats["band"] == "1-3 nights", "median_of_medians"
        ].iloc[0],
        df_band_stats.loc[
            df_band_stats["band"] == "4-6 nights", "median_of_medians"
        ].iloc[0],
        df_band_stats.loc[
            df_band_stats["band"] == "7-14 nights", "median_of_medians"
        ].iloc[0],
        df_band_stats.loc[
            df_band_stats["band"] == "15-29 nights", "median_of_medians"
        ].iloc[0],
    ]
) * 100.0

print("Consecutive-drop evidence (largest sequential declines):")
print(
    df_drop_ranking.head(12)
    .round({"median_booking_potential": 2, "median_drop_abs": 2, "median_drop_pct": 2})
    .to_string(index=False)
)
print("")
print("Grouped-band evidence (median booking-potential proxy by policy bands):")
print(
    df_band_stats[
        [
            "band",
            "listing_count",
            "share_of_listings_pct",
            "mean_median_booking_potential",
            "median_of_medians",
        ]
    ]
    .round(
        {
            "share_of_listings_pct": 2,
            "mean_median_booking_potential": 2,
            "median_of_medians": 2,
        }
    )
    .to_string(index=False)
)
print("")
print("Band-to-band median declines:")
print(
    df_band_comparison.round({"median_drop_abs": 2, "median_drop_pct": 2}).to_string(
        index=False
    )
)
print("")

# Concise conclusion based on the shape of the decline.
sharp_zone = "1-7 nights"
main_step = "1 -> 2 nights"
secondary_steps = "2 -> 3 and 3 -> 4 nights"

print(
    "Conclusion: the relationship is mostly a smooth monotonic decline, not a single discrete cliff. "
    f"The most defensible breakpoint zone is {sharp_zone}, with the strongest sequential drop at {main_step} "
    f"(-50.0%), followed by {secondary_steps} (-33.3% and -25.0%). "
    "A second, broader breakpoint is around 14 nights, where the median proxy falls from about 52.1 at 7 nights "
    "to 26.1 at 14 nights, and then to 12.2 at 30 nights. In practice, policy tiers of 7, 14, and 30 nights "
    "are the clearest defensible thresholds because they align with large cumulative declines while also matching "
    "substantial listing volume."
)


### 2.4: Quantify how much booking potential decreases when crossing selected global minimum_nights thresholds.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use the existing engineered dataframe; do not recreate or overwrite it.
df_threshold_source = df_airbnb_features.loc[
    pd.to_numeric(df_airbnb_features["minimum_nights_num"], errors="coerce") <= 60
].copy()

# Ensure the analysis columns are numeric and safe for aggregation.
df_threshold_source["minimum_nights_num"] = pd.to_numeric(
    df_threshold_source["minimum_nights_num"], errors="coerce"
)
df_threshold_source["booking_potential_proxy_cap_60"] = pd.to_numeric(
    df_threshold_source["booking_potential_proxy_cap_60"], errors="coerce"
)


def _group_stats(df_slice: pd.DataFrame) -> dict:
    series = df_slice["booking_potential_proxy_cap_60"].dropna()
    return {
        "n": int(series.shape[0]),
        "mean": series.mean(),
        "median": series.median(),
        "std": series.std(ddof=1),
    }


comparisons = [
    (
        "1 night",
        df_threshold_source["minimum_nights_num"].eq(1),
        "2+ nights",
        df_threshold_source["minimum_nights_num"].ge(2),
    ),
    (
        "2-3 nights",
        df_threshold_source["minimum_nights_num"].between(2, 3),
        "4-7 nights",
        df_threshold_source["minimum_nights_num"].between(4, 7),
    ),
    (
        "7-14 nights",
        df_threshold_source["minimum_nights_num"].between(7, 14),
        "15-30 nights",
        df_threshold_source["minimum_nights_num"].between(15, 30),
    ),
    (
        "<=7 nights",
        df_threshold_source["minimum_nights_num"].le(7),
        ">7 nights",
        df_threshold_source["minimum_nights_num"].gt(7),
    ),
    (
        "<=14 nights",
        df_threshold_source["minimum_nights_num"].le(14),
        ">14 nights",
        df_threshold_source["minimum_nights_num"].gt(14),
    ),
]

rows = []
for comparison_name, left_mask, right_name, right_mask in comparisons:
    df_left = df_threshold_source.loc[left_mask].copy()
    df_right = df_threshold_source.loc[right_mask].copy()

    left_stats = _group_stats(df_left)
    right_stats = _group_stats(df_right)

    abs_diff_mean = left_stats["mean"] - right_stats["mean"]
    abs_diff_median = left_stats["median"] - right_stats["median"]
    pct_diff_mean = (
        (abs_diff_mean / right_stats["mean"] * 100.0)
        if pd.notna(right_stats["mean"]) and right_stats["mean"] != 0
        else np.nan
    )
    pct_diff_median = (
        (abs_diff_median / right_stats["median"] * 100.0)
        if pd.notna(right_stats["median"]) and right_stats["median"] != 0
        else np.nan
    )

    rows.append(
        {
            "comparison": comparison_name,
            "left_group": comparison_name,
            "right_group": right_name,
            "left_n": left_stats["n"],
            "right_n": right_stats["n"],
            "left_mean_booking_potential": left_stats["mean"],
            "right_mean_booking_potential": right_stats["mean"],
            "left_median_booking_potential": left_stats["median"],
            "right_median_booking_potential": right_stats["median"],
            "left_std_booking_potential": left_stats["std"],
            "right_std_booking_potential": right_stats["std"],
            "mean_diff_abs": abs_diff_mean,
            "mean_diff_pct": pct_diff_mean,
            "median_diff_abs": abs_diff_median,
            "median_diff_pct": pct_diff_median,
        }
    )

df_threshold_effects = pd.DataFrame(rows)

# Rank by absolute median drop to highlight the most practical thresholds.
df_threshold_effects["median_drop_abs"] = df_threshold_effects["median_diff_abs"].abs()
df_threshold_effects["mean_drop_abs"] = df_threshold_effects["mean_diff_abs"].abs()
df_threshold_effects = df_threshold_effects.sort_values(
    ["median_drop_abs", "mean_drop_abs"], ascending=False
).reset_index(drop=True)

print("Global threshold comparisons for capped booking potential (<= 60 nights):")
print(
    df_threshold_effects[
        [
            "comparison",
            "left_n",
            "right_n",
            "left_mean_booking_potential",
            "right_mean_booking_potential",
            "left_median_booking_potential",
            "right_median_booking_potential",
            "left_std_booking_potential",
            "right_std_booking_potential",
            "mean_diff_abs",
            "mean_diff_pct",
            "median_diff_abs",
            "median_diff_pct",
        ]
    ]
    .round(2)
    .to_string(index=False)
)
print("")

largest_drop_row = df_threshold_effects.iloc[0]
print(
    f"Largest practical drop appears at {largest_drop_row['comparison']}, "
    f"with a median difference of {largest_drop_row['median_diff_abs']:.2f} "
    f"and a mean difference of {largest_drop_row['mean_diff_abs']:.2f}."
)
print(
    "The strongest threshold effects are typically the 1 vs 2-night split and the broader 7-night and 14-night breaks, "
    "with 7-14 vs 15-30 nights and <=14 vs >14 nights also showing large declines."
)

add_to_workspace(df_threshold_effects)


## Task 3: Assess how minimum stay tipping points and their impacts on booking potential vary by region, property/room type, and host segment, and synthesize findings to answer the user’s question about where cancellations or low bookings are likely to spike.

**Acceptance Criteria:** We have region- and segment-level summaries and at least one visualization showing how thresholds differ across cities and property/host segments, plus a printed synthesis linking these patterns back to the original question with clear, practical recommendations on minimum stay policies.

### 3.1: Compare booking potential across key minimum_nights thresholds by city to see how global patterns vary regionally.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use the existing engineered dataframe; do not recreate or overwrite it.
df_city_threshold_source = df_airbnb_features.loc[
    pd.to_numeric(df_airbnb_features["minimum_nights_num"], errors="coerce") <= 60
].copy()

df_city_threshold_source["minimum_nights_num"] = pd.to_numeric(
    df_city_threshold_source["minimum_nights_num"], errors="coerce"
)
df_city_threshold_source["booking_potential_proxy_cap_60"] = pd.to_numeric(
    df_city_threshold_source["booking_potential_proxy_cap_60"], errors="coerce"
)

# Ordered bands reflecting the global breakpoints.
band_categories = [
    "1 night",
    "2-3 nights",
    "4-7 nights",
    "8-14 nights",
    "15-30 nights",
    "31-60 nights",
]

df_city_threshold_source["minimum_stay_band"] = pd.cut(
    df_city_threshold_source["minimum_nights_num"],
    bins=[0, 1, 3, 7, 14, 30, 60],
    labels=band_categories,
    include_lowest=True,
    right=True,
    ordered=True,
)

# City-by-band summary.
df_city_thresholds = (
    df_city_threshold_source.groupby(["city", "minimum_stay_band"], as_index=False)
    .agg(
        listing_count=("listing_id", "count"),
        mean_booking_potential=("booking_potential_proxy_cap_60", "mean"),
        median_booking_potential=("booking_potential_proxy_cap_60", "median"),
    )
    .sort_values(["city", "minimum_stay_band"])
    .reset_index(drop=True)
)

# Make the band ordering explicit for downstream readability.
df_city_thresholds["minimum_stay_band"] = pd.Categorical(
    df_city_thresholds["minimum_stay_band"],
    categories=band_categories,
    ordered=True,
)

# Focus on the 5 largest cities and the most decision-relevant bands.
large_cities = ["Paris", "New York", "Sydney", "Rome", "Rio de Janeiro"]
df_city_thresholds_focus = df_city_thresholds.loc[
    df_city_thresholds["city"].isin(large_cities)
    & df_city_thresholds["minimum_stay_band"].isin(
        ["1 night", "2-3 nights", "4-7 nights", "8-14 nights", "15-30 nights"]
    )
].copy()

df_city_thresholds_focus["minimum_stay_band"] = pd.Categorical(
    df_city_thresholds_focus["minimum_stay_band"],
    categories=["1 night", "2-3 nights", "4-7 nights", "8-14 nights", "15-30 nights"],
    ordered=True,
)
df_city_thresholds_focus = df_city_thresholds_focus.sort_values(
    ["city", "minimum_stay_band"]
)

print("City-level minimum-stay threshold summary for the 5 largest cities:")
print(
    df_city_thresholds_focus[
        [
            "city",
            "minimum_stay_band",
            "listing_count",
            "mean_booking_potential",
            "median_booking_potential",
        ]
    ]
    .round({"mean_booking_potential": 2, "median_booking_potential": 2})
    .to_string(index=False)
)
print("")

# Compare steepest drops from 2-3 to 4-7 nights and from 8-14 to 15-30 nights.
df_city_drop_23_to_47 = df_city_thresholds.pivot(
    index="city", columns="minimum_stay_band", values="median_booking_potential"
).reset_index()
df_city_drop_23_to_47["drop_2_3_to_4_7_abs"] = (
    df_city_drop_23_to_47["2-3 nights"] - df_city_drop_23_to_47["4-7 nights"]
)
df_city_drop_23_to_47["drop_2_3_to_4_7_pct"] = (
    df_city_drop_23_to_47["drop_2_3_to_4_7_abs"]
    / df_city_drop_23_to_47["2-3 nights"]
    * 100.0
)

df_city_drop_814_to_1530 = df_city_thresholds.pivot(
    index="city", columns="minimum_stay_band", values="median_booking_potential"
).reset_index()
df_city_drop_814_to_1530["drop_8_14_to_15_30_abs"] = (
    df_city_drop_814_to_1530["8-14 nights"] - df_city_drop_814_to_1530["15-30 nights"]
)
df_city_drop_814_to_1530["drop_8_14_to_15_30_pct"] = (
    df_city_drop_814_to_1530["drop_8_14_to_15_30_abs"]
    / df_city_drop_814_to_1530["8-14 nights"]
    * 100.0
)

print(
    "Cities with the steepest drop from 2-3 nights to 4-7 nights (median booking potential):"
)
print(
    df_city_drop_23_to_47[["city", "drop_2_3_to_4_7_abs", "drop_2_3_to_4_7_pct"]]
    .sort_values(["drop_2_3_to_4_7_abs", "drop_2_3_to_4_7_pct"], ascending=False)
    .head(5)
    .round({"drop_2_3_to_4_7_abs": 2, "drop_2_3_to_4_7_pct": 2})
    .to_string(index=False)
)
print("")
print(
    "Cities with the steepest drop from 8-14 nights to 15-30 nights (median booking potential):"
)
print(
    df_city_drop_814_to_1530[
        ["city", "drop_8_14_to_15_30_abs", "drop_8_14_to_15_30_pct"]
    ]
    .sort_values(["drop_8_14_to_15_30_abs", "drop_8_14_to_15_30_pct"], ascending=False)
    .head(5)
    .round({"drop_8_14_to_15_30_abs": 2, "drop_8_14_to_15_30_pct": 2})
    .to_string(index=False)
)

add_to_workspace(df_city_thresholds)


### 3.2: Visually compare how booking potential at and above key minimum_nights thresholds differs across major cities.
_Output: figure_

In [ ]:
import pandas as pd
import plotly.express as px

# Use the existing city-level threshold summary; do not recreate or overwrite it.
df_city_threshold_plot = df_city_thresholds.loc[
    df_city_thresholds["city"].isin(
        ["Paris", "New York", "Sydney", "Rome", "Rio de Janeiro"]
    )
    & df_city_thresholds["minimum_stay_band"].isin(
        ["1 night", "2-3 nights", "4-7 nights", "8-14 nights", "15-30 nights"]
    )
].copy()

ordered_bands = ["1 night", "2-3 nights", "4-7 nights", "8-14 nights", "15-30 nights"]
df_city_threshold_plot["minimum_stay_band"] = pd.Categorical(
    df_city_threshold_plot["minimum_stay_band"],
    categories=ordered_bands,
    ordered=True,
)
df_city_threshold_plot = df_city_threshold_plot.sort_values(
    ["city", "minimum_stay_band"]
)

city_order = ["Paris", "New York", "Sydney", "Rome", "Rio de Janeiro"]
color_map = {
    "Paris": "#1f77b4",
    "New York": "#ff7f0e",
    "Sydney": "#2ca02c",
    "Rome": "#d62728",
    "Rio de Janeiro": "#9467bd",
}

fig = px.line(
    df_city_threshold_plot,
    x="minimum_stay_band",
    y="median_booking_potential",
    color="city",
    category_orders={"minimum_stay_band": ordered_bands, "city": city_order},
    color_discrete_map=color_map,
    markers=True,
    title="Booking-potential drop-off by minimum stay band and city",
    labels={
        "minimum_stay_band": "Minimum stay band",
        "median_booking_potential": "Median booking potential proxy",
        "city": "City",
    },
)
fig.update_traces(mode="lines+markers", marker_line_color="black", marker_line_width=1)
fig.update_yaxes(tickformat=",")
fig.show()


### 3.3: Assess whether superhosts, professional hosts, and different property/room types set different minimum_nights and experience different booking-potential impacts at key thresholds.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Use the existing engineered dataframe; do not recreate or overwrite it.
df_segment_source = df_airbnb_features.loc[
    pd.to_numeric(df_airbnb_features["minimum_nights_num"], errors="coerce") <= 60
].copy()

df_segment_source["minimum_nights_num"] = pd.to_numeric(
    df_segment_source["minimum_nights_num"], errors="coerce"
)
df_segment_source["booking_potential_proxy_cap_60"] = pd.to_numeric(
    df_segment_source["booking_potential_proxy_cap_60"], errors="coerce"
)

# Reuse the same ordered minimum-stay bands as the city analysis.
band_categories = [
    "1 night",
    "2-3 nights",
    "4-7 nights",
    "8-14 nights",
    "15-30 nights",
    "31-60 nights",
]

df_segment_source["minimum_stay_band"] = pd.cut(
    df_segment_source["minimum_nights_num"],
    bins=[0, 1, 3, 7, 14, 30, 60],
    labels=band_categories,
    include_lowest=True,
    right=True,
    ordered=True,
)


# Build host segment category from the superhost and professional-host flags.
def _segment_label(superhost_val, professional_val):
    is_superhost = pd.notna(superhost_val) and int(superhost_val) == 1
    is_professional = pd.notna(professional_val) and int(professional_val) == 1
    if is_superhost and is_professional:
        return "superhost_professional"
    if is_superhost and not is_professional:
        return "superhost_casual"
    if (not is_superhost) and is_professional:
        return "non_superhost_professional"
    return "non_superhost_casual"


df_segment_source["host_segment"] = [
    _segment_label(s, p)
    for s, p in zip(
        df_segment_source["is_superhost_flag"],
        df_segment_source["is_professional_host_flag"],
    )
]

df_segment_source["host_segment"] = pd.Categorical(
    df_segment_source["host_segment"],
    categories=[
        "superhost_professional",
        "superhost_casual",
        "non_superhost_professional",
        "non_superhost_casual",
    ],
    ordered=True,
)

df_segment_source["room_type"] = df_segment_source["room_type"].astype("string")

# Host segment by band summary.
df_host_segment_thresholds = (
    df_segment_source.groupby(["host_segment", "minimum_stay_band"], as_index=False)
    .agg(
        listing_count=("listing_id", "count"),
        mean_booking_potential=("booking_potential_proxy_cap_60", "mean"),
        median_booking_potential=("booking_potential_proxy_cap_60", "median"),
    )
    .assign(segment_family="host_segment")
)
df_host_segment_thresholds["minimum_stay_band"] = pd.Categorical(
    df_host_segment_thresholds["minimum_stay_band"],
    categories=band_categories,
    ordered=True,
)

# Room type by band summary.
df_room_type_thresholds = (
    df_segment_source.groupby(["room_type", "minimum_stay_band"], as_index=False)
    .agg(
        listing_count=("listing_id", "count"),
        mean_booking_potential=("booking_potential_proxy_cap_60", "mean"),
        median_booking_potential=("booking_potential_proxy_cap_60", "median"),
    )
    .assign(segment_family="room_type")
)
df_room_type_thresholds["minimum_stay_band"] = pd.Categorical(
    df_room_type_thresholds["minimum_stay_band"],
    categories=band_categories,
    ordered=True,
)

# Combine into one clear summary table.
df_segment_thresholds = pd.concat(
    [df_host_segment_thresholds, df_room_type_thresholds], ignore_index=True
).reset_index(drop=True)

df_segment_thresholds["segment_name"] = df_segment_thresholds["host_segment"].astype(
    "string"
)
df_segment_thresholds.loc[
    df_segment_thresholds["segment_family"].eq("room_type"), "segment_name"
] = df_segment_thresholds.loc[
    df_segment_thresholds["segment_family"].eq("room_type"), "room_type"
].astype(
    "string"
)

df_segment_thresholds = (
    df_segment_thresholds[
        [
            "segment_family",
            "segment_name",
            "minimum_stay_band",
            "listing_count",
            "mean_booking_potential",
            "median_booking_potential",
        ]
    ]
    .sort_values(["segment_family", "segment_name", "minimum_stay_band"])
    .reset_index(drop=True)
)

# Filtered view for key bands where counts are at least 100.
key_bands = ["1 night", "2-3 nights", "4-7 nights", "8-14 nights", "15-30 nights"]
df_segment_thresholds_focus = df_segment_thresholds.loc[
    df_segment_thresholds["minimum_stay_band"].isin(key_bands)
    & df_segment_thresholds["listing_count"].ge(100)
].copy()
df_segment_thresholds_focus = df_segment_thresholds_focus.sort_values(
    ["segment_family", "segment_name", "minimum_stay_band"]
)

print(
    "Host segment and room type threshold summary (filtered to key bands with at least 100 listings):"
)
print(
    df_segment_thresholds_focus[
        [
            "segment_family",
            "segment_name",
            "minimum_stay_band",
            "listing_count",
            "mean_booking_potential",
            "median_booking_potential",
        ]
    ]
    .round({"mean_booking_potential": 2, "median_booking_potential": 2})
    .to_string(index=False)
)
print("")

# Overall minimum-stay band distributions by host segment.
df_host_segment_band_dist = df_segment_source.groupby(
    ["host_segment", "minimum_stay_band"], as_index=False
).agg(listing_count=("listing_id", "count"))
df_host_segment_band_dist["share_pct_within_segment"] = (
    df_host_segment_band_dist["listing_count"]
    / df_host_segment_band_dist.groupby("host_segment")["listing_count"].transform(
        "sum"
    )
    * 100.0
)

print("Minimum-stay band distribution by host segment:")
print(
    df_host_segment_band_dist.sort_values(["host_segment", "minimum_stay_band"])
    .round({"share_pct_within_segment": 2})
    .to_string(index=False)
)
print("")

# Overall minimum-stay band distributions by room type.
df_room_type_band_dist = df_segment_source.groupby(
    ["room_type", "minimum_stay_band"], as_index=False
).agg(listing_count=("listing_id", "count"))
df_room_type_band_dist["share_pct_within_room_type"] = (
    df_room_type_band_dist["listing_count"]
    / df_room_type_band_dist.groupby("room_type")["listing_count"].transform("sum")
    * 100.0
)

print("Minimum-stay band distribution by room type:")
print(
    df_room_type_band_dist.sort_values(["room_type", "minimum_stay_band"])
    .round({"share_pct_within_room_type": 2})
    .to_string(index=False)
)
print("")

# Brief concentration note focused on higher minimum-stay bands.
df_high_band_host = df_host_segment_band_dist.loc[
    df_host_segment_band_dist["minimum_stay_band"].isin(
        ["15-30 nights", "31-60 nights"]
    )
].copy()
df_high_band_host = df_high_band_host.groupby("host_segment", as_index=False).agg(
    high_band_listings=("listing_count", "sum")
)
df_high_band_host["high_band_share_pct"] = (
    df_high_band_host["high_band_listings"]
    / df_host_segment_band_dist.groupby("host_segment")["listing_count"].sum().values
    * 100.0
)

print(
    "Higher-band concentration among host segments (15-30 and 31-60 nights combined):"
)
print(
    df_high_band_host.sort_values("high_band_share_pct", ascending=False)
    .round({"high_band_share_pct": 2})
    .to_string(index=False)
)

add_to_workspace(df_segment_thresholds)


### 3.4: Investigate why some regions or segments appear less affected by higher minimum stays, focusing on price, amenity richness, and review quality as potential moderators.
_Output: print_

In [ ]:
import numpy as np
import pandas as pd

# Build a temporary diagnostic summary; do not overwrite existing workspace dataframes.
df_diagnostic_source = df_airbnb_features.loc[
    pd.to_numeric(df_airbnb_features["minimum_nights_num"], errors="coerce") <= 60
].copy()

df_diagnostic_source["minimum_nights_num"] = pd.to_numeric(
    df_diagnostic_source["minimum_nights_num"], errors="coerce"
)
df_diagnostic_source["price"] = pd.to_numeric(
    df_diagnostic_source["price"], errors="coerce"
)
df_diagnostic_source["amenity_count"] = pd.to_numeric(
    df_diagnostic_source["amenity_count"], errors="coerce"
)
df_diagnostic_source["review_scores_rating"] = pd.to_numeric(
    df_diagnostic_source["review_scores_rating"], errors="coerce"
)

band_categories = ["2-3 nights", "4-7 nights", "8-14 nights", "15-30 nights"]
df_diagnostic_source["minimum_stay_band"] = pd.cut(
    df_diagnostic_source["minimum_nights_num"],
    bins=[1, 3, 7, 14, 30],
    labels=band_categories,
    include_lowest=True,
    right=True,
    ordered=True,
)

focus_cities = ["Paris", "Mexico City", "Sydney", "Rio de Janeiro"]
df_diagnostic_table = (
    df_diagnostic_source.loc[
        df_diagnostic_source["city"].isin(focus_cities)
        & df_diagnostic_source["minimum_stay_band"].isin(band_categories)
    ]
    .groupby(["city", "minimum_stay_band"], as_index=False)
    .agg(
        listing_count=("listing_id", "count"),
        median_price=("price", "median"),
        median_amenity_count=("amenity_count", "median"),
        review_coverage_rate=(
            "review_scores_rating",
            lambda s: s.notna().mean() * 100.0,
        ),
        median_review_score=("review_scores_rating", "median"),
    )
    .reset_index(drop=True)
)

df_diagnostic_table["minimum_stay_band"] = pd.Categorical(
    df_diagnostic_table["minimum_stay_band"],
    categories=band_categories,
    ordered=True,
)
df_diagnostic_table = df_diagnostic_table.sort_values(
    ["city", "minimum_stay_band"]
).reset_index(drop=True)

print(
    "City diagnostic table: price, amenities, and review coverage by minimum-stay band"
)
print(
    df_diagnostic_table[
        [
            "city",
            "minimum_stay_band",
            "listing_count",
            "median_price",
            "median_amenity_count",
            "review_coverage_rate",
            "median_review_score",
        ]
    ]
    .round(
        {
            "median_price": 2,
            "median_amenity_count": 2,
            "review_coverage_rate": 2,
            "median_review_score": 2,
        }
    )
    .to_string(index=False)
)
print("")

# Merge-based comparison of 15-30 nights vs 2-3 nights within each city.
df_compare_base = df_diagnostic_table.loc[
    df_diagnostic_table["minimum_stay_band"].isin(["2-3 nights", "15-30 nights"])
].copy()

df_compare_base = df_compare_base[
    [
        "city",
        "minimum_stay_band",
        "listing_count",
        "median_price",
        "median_amenity_count",
        "review_coverage_rate",
        "median_review_score",
    ]
].copy()

df_compare_wide = (
    df_compare_base[df_compare_base["minimum_stay_band"] == "2-3 nights"]
    .merge(
        df_compare_base[df_compare_base["minimum_stay_band"] == "15-30 nights"],
        on="city",
        how="outer",
        suffixes=("_2-3 nights", "_15-30 nights"),
    )
    .reset_index(drop=True)
)

for metric in [
    "listing_count",
    "median_price",
    "median_amenity_count",
    "review_coverage_rate",
    "median_review_score",
]:
    left_col = f"{metric}_15-30 nights"
    right_col = f"{metric}_2-3 nights"
    df_compare_wide[f"{metric}_diff_abs"] = (
        df_compare_wide[left_col] - df_compare_wide[right_col]
    )

print("Comparison of 15-30 nights versus 2-3 nights within each city")
print(
    df_compare_wide[
        [
            "city",
            "listing_count_diff_abs",
            "median_price_diff_abs",
            "median_amenity_count_diff_abs",
            "review_coverage_rate_diff_abs",
            "median_review_score_diff_abs",
        ]
    ]
    .round(
        {
            "listing_count_diff_abs": 0,
            "median_price_diff_abs": 2,
            "median_amenity_count_diff_abs": 2,
            "review_coverage_rate_diff_abs": 2,
            "median_review_score_diff_abs": 2,
        }
    )
    .to_string(index=False)
)


### 3.5: Validate internal consistency of findings and synthesize a direct answer to the question about where minimum stay lengths cause spikes in low booking potential across regions.
_Output: print_

_No successful execution recorded for this subtask._